# Paper 2 · Langkah 6: uji ketahanan

Tiga uji untuk memastikan hasil langkah 4 dan 5 tidak rapuh:

1. **Occlusion pada skala logit.** Di langkah 4, occlusion diukur dari turunnya probabilitas. Karena model sangat yakin, angkanya jenuh dan 5 sampai 10% peta kosong. Di sini occlusion diulang pada skala logit dengan presisi penuh, lalu uji deletion dan statistik elemen desain dihitung ulang.
2. **Uji tanpa warna dan tanpa detail.** Model diuji pada gambar hitam-putih, gambar dengan saturasi separuh, dan gambar yang diburamkan. Tujuannya melihat apakah model bergantung pada warna atau pada detail halus.
3. **Baseline ukuran desain.** Seberapa jauh sebelas ukuran desain saja (tanpa jaringan saraf) bisa membedakan gambar AI dan karya manusia.

**Sebelum Run all:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Hasil tiap bagian langsung disimpan ke Drive. Kalau sesi terputus, klik Run all lagi; bagian yang sudah selesai dilewati.

## 1. Persiapan dan cek GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, io, json, time, shutil, zipfile, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision.transforms.functional import gaussian_blur
import cv2, scipy, sklearn
from scipy import stats, ndimage
from PIL import Image
import PIL

# Pengaturan (sama dengan langkah 4 dan 5; jangan diubah tanpa mencatatnya di log kerja)
SEED = 42
INPUT = 224
PATCH, STRIDE = 32, 16
TOP = 0.20
BLUR_KERNEL, BLUR_SIGMA = 21, 7.0
HUE_BINS, GLCM_LEVELS = 36, 32
CANNY_LOW, CANNY_HIGH = 100, 200
CHROMA_MIN = 0.10

PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
MODELS = RESULTS / 'models'
MAPS = RESULTS / 'peta'
MAPS2 = RESULTS / 'peta_logit'
ELEM2 = RESULTS / 'elemen_logit'
CACHE = PROJECT / 'data_olahan'
LOCAL = Path('/content/work')
for p in [MAPS2, ELEM2, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'TIDAK ADA'
print('GPU:', gpu)
assert device == 'cuda', 'GPU belum aktif. Buka Runtime → Change runtime type → pilih T4 GPU → Save, lalu Run all lagi.'
USE_AMP = device == 'cuda'
torch.backends.cudnn.benchmark = True

env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\ntorch: {torch.__version__}\n'
       f'torchvision: {torchvision.__version__}\nnumpy: {np.__version__}\nscipy: {scipy.__version__}\nsklearn: {sklearn.__version__}\n'
       f'opencv: {cv2.__version__}\npillow: {PIL.__version__}\ngpu: {gpu}\n'
       f'occlusion logit: kotak {PATCH}, langkah {STRIDE}, presisi penuh (float32, tanpa autocast)\n')
(NOTES / 'lingkungan_langkah6.txt').write_text(env)
print(env)

## 2. Data

In [ ]:
manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
assert len(manifest) == 3000 and 'fold' in manifest.columns, 'Jalankan notebook langkah 2 dulu.'
manifest['y'] = (manifest['label'] == 'ai').astype(int)
assert (RESULTS / 'rq2_elemen_ringkasan.csv').exists(), 'Jalankan notebook langkah 5 dulu.'

local_raw = LOCAL / 'raw'
raw_zip = CACHE / 'ai_artbench_sample_raw.zip'
t0 = time.time()
if not local_raw.exists():
    shutil.copy(raw_zip, LOCAL / 'raw.zip')
    with zipfile.ZipFile(LOCAL / 'raw.zip') as z:
        z.extractall(local_raw)
manifest['raw_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)

def to_norm(im):
    im = im.convert('RGB')
    w, h = im.size
    if w != h:
        s = min(w, h); left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    buf = io.BytesIO()
    im.save(buf, 'JPEG', quality=90, subsampling=0, optimize=False)
    return buf.getvalue()

def make_condition(row):
    with Image.open(row.raw_path) as im:
        im.load()
        if row.source == 'human':
            buf = io.BytesIO()
            im.convert('RGB').save(buf, 'JPEG', quality=75, subsampling=2, optimize=False)
            im = Image.open(io.BytesIO(buf.getvalue())); im.load()
        return Image.open(io.BytesIO(to_norm(im))).convert('RGB')

def _one(row):
    im = make_condition(row)
    if im.size != (INPUT, INPUT):
        im = im.resize((INPUT, INPUT), Image.BICUBIC)
    return np.asarray(im, dtype=np.uint8)
with ThreadPoolExecutor(8) as ex:
    XH = np.stack(list(ex.map(_one, manifest.itertuples())))        # kondisi riwayat_disamakan, N,H,W,3
XT = torch.from_numpy(XH).permute(0, 3, 1, 2).contiguous()           # N,3,H,W
idx_of = {s: i for i, s in enumerate(manifest['sample_id'])}
src_all, folds, y_all = manifest['source'].values, manifest['fold'].values, manifest['y'].values
print(f'Data siap ({time.time() - t0:.0f} detik):', XH.shape)

## 3. Fungsi model dan peta

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
NPIX = INPUT * INPUT
FRACS = torch.arange(1, 11, device=device) * 0.05

def norm_t(xb):
    return ((xb.to(device).float().div(255) - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def load_model(name):
    m = torchvision.models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 2)
    m.load_state_dict(torch.load(MODELS / f'{name}.pt', map_location=device))
    return m.to(device).to(memory_format=torch.channels_last).eval()

@torch.no_grad()
def probs(model, xn, bs=256):
    out = []
    for i in range(0, len(xn), bs):
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(xn[i:i + bs])
        out.append(torch.softmax(logits.float(), 1))
    return torch.cat(out)

@torch.no_grad()
def logits32(model, xn, bs=256):
    return torch.cat([model(xn[i:i + bs]).float() for i in range(0, len(xn), bs)])     # presisi penuh, tanpa autocast

pos = list(range(0, INPUT - PATCH + 1, STRIDE)); N1 = len(pos)
M1 = torch.zeros(N1 * N1, 1, INPUT, INPUT, dtype=torch.bool, device=device); k = 0
for i in pos:
    for j in pos:
        M1[k, 0, i:i + PATCH, j:j + PATCH] = True; k += 1

@torch.no_grad()
def occlusion_logit(model, xn, pred):
    """Turunnya selisih logit (kelas tebakan dikurangi kelas lain) saat tiap kotak ditutup: (B,n,n) dan selisih awal (B,)."""
    out, m0s = [], []
    for i in range(len(xn)):
        x = xn[i:i + 1]; c = int(pred[i]); mu = x.mean((2, 3), keepdim=True)
        z0 = logits32(model, x)[0]; m0 = z0[c] - z0[1 - c]
        z = logits32(model, torch.where(M1, mu, x))
        out.append((m0 - (z[:, c] - z[:, 1 - c])).reshape(N1, N1)); m0s.append(m0)
    return torch.stack(out), torch.stack(m0s)

def patch_to_cells(d):
    kk, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C, device=d.device); cnt = torch.zeros(C, C, device=d.device)
    for a in range(kk):
        for b in range(kk):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def rank_desc(maps, gen):
    flat = maps.reshape(len(maps), -1)
    flat = flat + 1e-7 * flat.abs().amax(1, keepdim=True).clamp(min=1e-12) * torch.rand(flat.shape, device=flat.device, generator=gen)
    order = flat.argsort(1, descending=True)
    rank = torch.empty_like(order)
    rank.scatter_(1, order, torch.arange(order.shape[1], device=flat.device).expand_as(order))
    return rank

def spearman_t(a, b, gen):
    ra, rb = rank_desc(a, gen).float(), rank_desc(b, gen).float()
    ra, rb = ra - ra.mean(1, keepdim=True), rb - rb.mean(1, keepdim=True)
    rho = (ra * rb).sum(1) / (ra.norm(dim=1) * rb.norm(dim=1)).clamp(min=1e-12)
    const = (a.reshape(len(a), -1).std(1) < 1e-12) | (b.reshape(len(b), -1).std(1) < 1e-12)
    rho[const] = float('nan')
    return rho

@torch.no_grad()
def deletion_one(model, x, m224, c, p0, gen):
    rank = rank_desc(m224[None], gen)
    thr = (FRACS * NPIX).long()
    mask = (rank[:, None, :] < thr[None, :, None]).reshape(-1, 1, INPUT, INPUT)
    out = []
    for rep in (gaussian_blur(x, [BLUR_KERNEL, BLUR_KERNEL], [BLUR_SIGMA, BLUR_SIGMA]), x.mean((2, 3), keepdim=True)):
        p = probs(model, torch.where(mask, rep, x))[:, c]
        out.append(float((0.5 * p0 + p[:-1].sum() + 0.5 * p[-1]) / len(FRACS)))
    return out
print('Fungsi siap. Posisi kotak occlusion:', N1, 'x', N1)

## 4. Uji 1a: occlusion pada skala logit (bagian terlama)

Tujuh bagian: lima fold kondisi riwayat_disamakan dan dua model lintas generator. Untuk setiap gambar dihitung peta baru, kesepakatannya dengan peta lama, dan uji deletion untuk peta baru.

In [ ]:
UNITS = [f'riwayat_disamakan_fold{k}' for k in range(5)] + ['lintas2_riwayat_disamakan_latihLD_ujiSD', 'lintas2_riwayat_disamakan_latihSD_ujiLD']

def process_maps(name):
    f = MAPS2 / f'{name}.npz'
    if f.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    t0 = time.time()
    z = np.load(MAPS / f'{name}.npz', allow_pickle=True)
    ids = z['sample_id'].astype(str); idx = np.array([idx_of[s] for s in ids]); pred = z['pred'].astype(int); n = len(idx)
    p_pred = z['p'][np.arange(n), pred]; occ_old = torch.from_numpy(z['occ'][np.arange(n), pred]).float().to(device)
    model = load_model(name)
    gen = torch.Generator(device=device); gen.manual_seed(SEED + sum(map(ord, name)))
    occ_new, m0s, rho, iou, dele = [], [], [], [], []
    for s in range(0, n, 16):
        b = idx[s:s + 16]; xn = norm_t(XT[b])
        d, m0 = occlusion_logit(model, xn, pred[s:s + 16])
        c_new = patch_to_cells(d).clamp(min=0); c_old = patch_to_cells(occ_old[s:s + 16]).clamp(min=0)
        rho.append(spearman_t(c_old, c_new, gen).cpu())
        u_new, u_old = up(c_new), up(c_old)
        Rn = rank_desc(u_new, gen) < TOP * NPIX; Ro = rank_desc(u_old, gen) < TOP * NPIX
        iou.append(((Rn & Ro).sum(1) / (Rn | Ro).sum(1)).float().cpu())
        for i in range(len(b)):
            dele.append(deletion_one(model, xn[i:i + 1], u_new[i], int(pred[s + i]), float(p_pred[s + i]), gen))
        occ_new.append(d.cpu()); m0s.append(m0.cpu())
    occ_new = torch.cat(occ_new).numpy().astype(np.float32); dele = np.array(dele, dtype=np.float32)
    np.savez_compressed(f, sample_id=ids, pred=pred, margin0=torch.cat(m0s).numpy().astype(np.float32), occ_logit=occ_new,
                        rho_lama_baru=torch.cat(rho).numpy(), iou_lama_baru=torch.cat(iou).numpy(),
                        del_blur_occlogit=dele[:, 0], del_mean_occlogit=dele[:, 1])
    print(f'{name}: {n} gambar, {time.time() - t0:.0f} detik')
    del model; torch.cuda.empty_cache()

for name in UNITS:
    process_maps(name)

## 5. Uji 2: tanpa warna dan tanpa detail

Lima model kondisi riwayat_disamakan diuji pada fold ujinya masing-masing, dengan gambar yang diubah:
- **hitam-putih**: warna dibuang seluruhnya.
- **saturasi 50%**: warna dilemahkan separuh.
- **buram ringan / buram kuat**: detail halus dibuang, warna dan susunan besar dipertahankan.

Catatan baca: gambar yang diubah berbeda dari gambar latihan, jadi penurunan kinerja sebagian wajar. Yang dibandingkan adalah **seberapa besar** penurunan antar jenis perubahan.

In [ ]:
from sklearn.metrics import roc_auc_score

def transform(xb, kind):
    x = xb.to(device).float()
    gray = (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]).expand(-1, 3, -1, -1)
    if kind == 'hitam-putih': x = gray
    elif kind == 'saturasi 50%': x = 0.5 * x + 0.5 * gray
    elif kind == 'buram ringan': x = gaussian_blur(x, [13, 13], [2.0, 2.0])
    elif kind == 'buram kuat': x = gaussian_blur(x, [25, 25], [4.0, 4.0])
    return ((x.clamp(0, 255) / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)

KINDS = ['asli', 'hitam-putih', 'saturasi 50%', 'buram ringan', 'buram kuat']
abl_file = RESULTS / 'rq1_uji_warna_detail_per_fold.csv'
if abl_file.exists():
    A = pd.read_csv(abl_file); print('Uji warna dan detail: sudah selesai, dilewati.')
else:
    rows = []
    for k in range(5):
        model = load_model(f'riwayat_disamakan_fold{k}'); te = np.where(folds == k)[0]
        for kind in KINDS:
            p = torch.cat([probs(model, transform(XT[te[i:i + 128]], kind)) for i in range(0, len(te), 128)])[:, 1].cpu().numpy()
            pred = (p >= 0.5).astype(int); yt = y_all[te]; s = src_all[te]
            rows.append({'fold': k, 'perubahan': kind, 'akurasi': (pred == yt).mean(), 'auc': roc_auc_score(yt, p),
                         'auc_manusia_vs_LD': roc_auc_score(yt[s != 'stable_diffusion'], p[s != 'stable_diffusion']),
                         'auc_manusia_vs_SD': roc_auc_score(yt[s != 'latent_diffusion'], p[s != 'latent_diffusion']),
                         'benar_manusia': (pred[s == 'human'] == 0).mean(), 'benar_LD': (pred[s == 'latent_diffusion'] == 1).mean(),
                         'benar_SD': (pred[s == 'stable_diffusion'] == 1).mean()})
        del model; torch.cuda.empty_cache()
    A = pd.DataFrame(rows); A.to_csv(abl_file, index=False)
T = A.groupby('perubahan')[['akurasi', 'auc', 'auc_manusia_vs_LD', 'auc_manusia_vs_SD', 'benar_manusia', 'benar_LD', 'benar_SD']].mean().reindex(KINDS)
T.to_csv(RESULTS / 'rq1_uji_warna_detail.csv')
print('\nW. UJI TANPA WARNA DAN TANPA DETAIL (rata-rata 5 fold)')
print(T.round(3).to_string())

## 6. Uji 1b: elemen desain dihitung ulang dengan peta logit

In [ ]:
NPIX = INPUT * INPUT
MEASURES = ['hue_entropy', 'saturation', 'colourfulness', 'brightness', 'contrast', 'gradient',
            'edge_density', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']
LABEL = {'hue_entropy': 'Hue: entropi', 'saturation': 'Warna: saturasi', 'colourfulness': 'Warna: kekayaan',
         'brightness': 'Nada: kecerahan', 'contrast': 'Nada: kontras', 'gradient': 'Nada: gradasi',
         'edge_density': 'Garis: kepadatan tepi', 'glcm_contrast': 'Tekstur: kontras',
         'glcm_homogeneity': 'Tekstur: homogenitas', 'glcm_energy': 'Tekstur: energi', 'glcm_correlation': 'Tekstur: korelasi'}

# Pasangan piksel bertetangga untuk GLCM (4 arah, jarak 1)
_flat = np.arange(NPIX).reshape(INPUT, INPUT)
PAIRS = []
for dy, dx in [(0, 1), (1, 0), (1, 1), (1, -1)]:
    a = _flat[max(0, -dy):INPUT - max(0, dy), max(0, -dx):INPUT - max(0, dx)]
    b = _flat[max(0, dy):INPUT - max(0, -dy), max(0, dx):INPUT - max(0, -dx)]
    PAIRS.append((a.ravel(), b.ravel()))
_lv = np.arange(GLCM_LEVELS, dtype=np.float64)
_I, _J = np.meshgrid(_lv, _lv, indexing='ij')
_D2 = (_I - _J) ** 2

def precompute(img):
    f = {}
    rgb = img.astype(np.float32)
    hsv = cv2.cvtColor(rgb / 255.0, cv2.COLOR_RGB2HSV)            # H 0-360, S 0-1, V 0-1
    f['hbin'] = np.minimum((hsv[..., 0] / 360.0 * HUE_BINS).astype(np.int64), HUE_BINS - 1).ravel()
    f['sat'] = hsv[..., 1].ravel(); f['val'] = hsv[..., 2].ravel()
    f['chroma_ok'] = (f['sat'] > CHROMA_MIN) & (f['val'] > CHROMA_MIN)
    f['rg'] = (rgb[..., 0] - rgb[..., 1]).ravel()
    f['yb'] = (0.5 * (rgb[..., 0] + rgb[..., 1]) - rgb[..., 2]).ravel()
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f['lum'] = gray.astype(np.float32).ravel()
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    f['grad'] = np.sqrt(gx ** 2 + gy ** 2).ravel()
    f['edge'] = (cv2.Canny(gray, CANNY_LOW, CANNY_HIGH) > 0).ravel()
    q = (gray.astype(np.int64) * GLCM_LEVELS // 256).ravel()
    f['codes'] = [q[a] * GLCM_LEVELS + q[b] for a, b in PAIRS]
    return f

def region_measures(f, m):
    out = dict.fromkeys(MEASURES, np.nan)
    if m.sum() < 100:
        return out
    sel = m & f['chroma_ok']
    if sel.sum() >= 50:
        h = np.bincount(f['hbin'][sel], minlength=HUE_BINS).astype(np.float64); p = h / h.sum(); p = p[p > 0]
        out['hue_entropy'] = float(-(p * np.log2(p)).sum())
    out['saturation'] = float(f['sat'][m].mean())
    rg, yb = f['rg'][m], f['yb'][m]
    out['colourfulness'] = float(np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))
    out['brightness'] = float(f['val'][m].mean())
    out['contrast'] = float(f['lum'][m].std())
    out['gradient'] = float(f['grad'][m].mean())
    out['edge_density'] = float(f['edge'][m].mean())
    P = np.zeros(GLCM_LEVELS * GLCM_LEVELS)
    for (a, b), code in zip(PAIRS, f['codes']):
        ok = m[a] & m[b]
        P += np.bincount(code[ok], minlength=GLCM_LEVELS * GLCM_LEVELS)
    if P.sum() > 0:
        P = P.reshape(GLCM_LEVELS, GLCM_LEVELS); P = P + P.T; P /= P.sum()
        out['glcm_contrast'] = float((P * _D2).sum())
        out['glcm_homogeneity'] = float((P / (1 + _D2)).sum())
        out['glcm_energy'] = float(np.sqrt((P ** 2).sum()))
        mu = (P.sum(1) * _lv).sum(); var = (P.sum(1) * (_lv - mu) ** 2).sum()
        out['glcm_correlation'] = float((P * (_I - mu) * (_J - mu)).sum() / var) if var > 1e-12 else np.nan
    return out

GRID = max(1, INPUT // 32)
def composition(m):
    m2 = m.reshape(INPUT, INPUT); ys, xs = np.nonzero(m2); c = (INPUT - 1) / 2
    cell = m2[:GRID * (INPUT // GRID), :GRID * (INPUT // GRID)].reshape(GRID, INPUT // GRID, GRID, INPUT // GRID).sum((1, 3)).ravel().astype(np.float64)
    p = cell / cell.sum(); p = p[p > 0]
    lab, ncomp = ndimage.label(m2, structure=np.ones((3, 3)))
    sizes = np.bincount(lab.ravel())[1:]
    return {'comp_centroid_dist': float(np.sqrt((ys.mean() - c) ** 2 + (xs.mean() - c) ** 2) / (c * 2 ** 0.5)),
            'comp_spatial_entropy': float(-(p * np.log2(p)).sum() / np.log2(GRID * GRID)) if GRID > 1 else np.nan,
            'comp_n_components': int(ncomp), 'comp_largest_share': float(sizes.max() / sizes.sum())}

def patch_to_cells_cpu(d):
    k, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C); cnt = torch.zeros(C, C)
    for a in range(k):
        for b in range(k):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def rank_maps(maps, rng):
    n = len(maps); rank = np.empty((n, NPIX), dtype=np.uint16)
    for s in range(0, n, 200):
        flat = maps[s:s + 200].reshape(-1, NPIX).astype(np.float64)
        flat = flat + 1e-9 * rng.random(flat.shape)                 # pemecah nilai kembar
        order = np.argsort(-flat, axis=1)
        np.put_along_axis(rank[s:s + 200], order, np.arange(NPIX, dtype=np.uint16)[None].repeat(len(order), 0), axis=1)
    return rank
ALL = np.ones(NPIX, dtype=bool)

def process_elem(name):
    out_file = ELEM2 / f'{name}.csv'
    if out_file.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    t0 = time.time()
    z = np.load(MAPS2 / f'{name}.npz', allow_pickle=True)
    ids = z['sample_id'].astype(str); idx = np.array([idx_of[s] for s in ids]); pred = z['pred'].astype(int); n = len(idx)
    rng = np.random.default_rng(SEED + sum(map(ord, name)))
    occ = torch.from_numpy(z['occ_logit']).float(); cells = patch_to_cells_cpu(occ).clamp(min=0)
    empty = (cells.amax((1, 2)) <= 0).numpy(); ranks = rank_maps(up(cells).numpy(), rng)
    partner = np.arange(n); s_here = src_all[idx]
    for s in np.unique(s_here):
        loc = rng.permutation(np.where(s_here == s)[0])
        if len(loc) > 1:
            partner[loc] = np.roll(loc, -1)
    rows = []
    for k in range(n):
        row = {'bagian': name, 'sample_id': ids[k], 'pred': int(pred[k]), 'peta_kosong': bool(empty[k]),
               'margin0': float(z['margin0'][k]), 'occ_max_drop_logit': float(occ[k].max()),
               'rho_lama_baru': float(z['rho_lama_baru'][k]), 'iou_lama_baru': float(z['iou_lama_baru'][k]),
               'del_blur_occlogit': float(z['del_blur_occlogit'][k]), 'del_mean_occlogit': float(z['del_mean_occlogit'][k])}
        if not empty[k]:
            f = precompute(XH[idx[k]]); m = ranks[k] < TOP * NPIX
            for tag, mask in (('in', m), ('out', ~m)):
                row.update({f'{tag}_{a}': v for a, v in region_measures(f, mask).items()})
            if partner[k] != k and not empty[partner[k]]:
                ms = ranks[partner[k]] < TOP * NPIX
                for tag, mask in (('shin', ms), ('shout', ~ms)):
                    row.update({f'{tag}_{a}': v for a, v in region_measures(f, mask).items()})
        rows.append(row)
    pd.DataFrame(rows).to_csv(out_file, index=False)
    print(f'{name}: {n} gambar, {time.time() - t0:.0f} detik')

for name in UNITS:
    process_elem(name)

## 7. Ringkasan ketiga uji

- **K1 sampai K4**: apakah peta logit memberi hasil yang sama dengan peta lama.
- **D**: baseline ukuran desain (regresi logistik pada sebelas ukuran seluruh gambar).

In [ ]:
SRC = {'human': 'manusia', 'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
df = pd.concat([pd.read_csv(f) for f in sorted(ELEM2.glob('*.csv'))], ignore_index=True)
df = df.join(manifest.set_index('sample_id')[['source', 'style', 'fold', 'y']], on='sample_id')
df['lintas'] = df.bagian.str.startswith('lintas2')
df['benar'] = df.pred == df.y
def glabel(r):
    s = SRC[r['source']]
    if r['lintas']:
        return f'lintas: {s} lolos' if (r['y'] == 1 and r['pred'] == 0) else (f'lintas: {s} benar' if r['benar'] else f'lintas: {s} dicap AI')
    return f'5-fold: {s} benar' if r['benar'] else f'5-fold: {s} salah'
df['kelompok'] = df.apply(glabel, axis=1)
df['model'] = np.where(df.bagian.str.contains('latihLD'), 'model-LD', np.where(df.bagian.str.contains('latihSD'), 'model-SD', '5-fold'))
df.to_csv(RESULTS / 'rq2_elemen_logit_per_gambar.csv', index=False)
print('Bagian selesai:', df.bagian.nunique(), 'dari 7 | baris:', len(df))

MAIN = ['5-fold: manusia benar', '5-fold: LD benar', '5-fold: SD benar', 'lintas: LD lolos', 'lintas: SD lolos', 'lintas: manusia benar']
SHORT = {'5-fold: manusia benar': 'man-5f', '5-fold: LD benar': 'LD-5f', '5-fold: SD benar': 'SD-5f',
         'lintas: LD lolos': 'LD-lolos', 'lintas: SD lolos': 'SD-lolos', 'lintas: manusia benar': 'man-lin'}

# K1: peta kosong dan kesepakatan peta lama (probabilitas) dengan peta baru (logit)
k1 = df[df.kelompok.isin(MAIN)].groupby('kelompok').agg(n=('sample_id', 'size'), peta_kosong=('peta_kosong', 'sum'),
        median_selisih_logit_awal=('margin0', 'median'), median_turun_logit_terbesar=('occ_max_drop_logit', 'median'),
        median_korelasi_lama_baru=('rho_lama_baru', 'median'), median_iou_lama_baru=('iou_lama_baru', 'median')).reindex(MAIN)
print('\nK1. PETA LOGIT: jumlah kosong, besar sinyal, dan kesepakatan dengan peta lama')
print(k1.round(3).to_string())

# K2: uji deletion, peta lama vs peta baru vs pembanding (dari langkah 4)
old = pd.read_csv(RESULTS / 'rq2_peta_per_gambar.csv')
old = old[old.bagian.isin(UNITS)][['bagian', 'sample_id', 'del_blur_gc', 'del_blur_occ', 'del_blur_centre', 'del_blur_random',
                                   'del_mean_gc', 'del_mean_occ', 'del_mean_centre', 'del_mean_random']]
dd = df.merge(old, on=['bagian', 'sample_id'], how='left')
k2 = dd[dd.kelompok.isin(MAIN)].groupby('kelompok')[['del_blur_occlogit', 'del_blur_occ', 'del_blur_gc', 'del_blur_centre', 'del_blur_random',
        'del_mean_occlogit', 'del_mean_occ', 'del_mean_gc', 'del_mean_centre', 'del_mean_random']].mean().reindex(MAIN)
k2.columns = ['blur:logit', 'blur:prob', 'blur:gradcam', 'blur:tengah', 'blur:acak', 'rata:logit', 'rata:prob', 'rata:gradcam', 'rata:tengah', 'rata:acak']
k2.to_csv(RESULTS / 'rq2_ketahanan_deletion.csv')
print('\nK2. UJI DELETION (makin kecil makin setia). logit = occlusion baru; prob = occlusion lama')
print(k2.round(3).to_string())

def paired_effect(x):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; x = x[x != 0]
    if len(x) < 10:
        return np.nan, np.nan, len(x)
    r = stats.rankdata(np.abs(x)); pos_, neg_ = r[x > 0].sum(), r[x < 0].sum()
    return (pos_ - neg_) / (pos_ + neg_), stats.wilcoxon(x).pvalue, len(x)
def bh(p):
    p = np.asarray(p, dtype=float); out = np.full(len(p), np.nan); ok = ~np.isnan(p)
    if ok.sum() == 0: return out
    pv = p[ok]; order = np.argsort(pv); ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]; res = np.empty(len(pv)); res[order] = np.minimum(ranked, 1)
    out[ok] = res; return out

def effects(d, key):
    rows = []
    for grp, g in d[~d.peta_kosong].groupby(key):
        for m in MEASURES:
            r, p, n = paired_effect((g[f'in_{m}'] - g[f'out_{m}']) - (g[f'shin_{m}'] - g[f'shout_{m}']))
            rows.append({'kelompok': grp, 'ukuran': m, 'n': n, 'r': r, 'p': p})
    S = pd.DataFrame(rows); S['p_fdr'] = bh(S.p.values); S['bermakna'] = S.p_fdr < 0.05
    return S

S_new = effects(df[df.kelompok.isin(MAIN)], 'kelompok')
S_old = pd.read_csv(RESULTS / 'rq2_elemen_ringkasan.csv')
S_old = S_old[(S_old.metode == 'occlusion') & np.isclose(S_old.ambang, 0.2) & S_old.kelompok.isin(MAIN)][['kelompok', 'ukuran', 'r', 'bermakna']]
C = S_new.merge(S_old, on=['kelompok', 'ukuran'], suffixes=('_logit', '_prob'))
C.to_csv(RESULTS / 'rq2_ketahanan_elemen.csv', index=False)

def cell(r, sig):
    return ('nan' if pd.isna(r) else f'{r:+.2f}') + ('*' if sig else ' ')
print('\nK3. ELEMEN DESAIN, occlusion 20%: r peta logit | r peta lama (* = bermakna)')
C['sel'] = [cell(a, b) + '|' + cell(c, d) for a, b, c, d in zip(C.r_logit, C.bermakna_logit, C.r_prob, C.bermakna_prob)]
pv = C.pivot(index='ukuran', columns='kelompok', values='sel').reindex(MEASURES)[[g for g in MAIN if g in C.kelompok.unique()]]
pv.index = [LABEL[i] for i in pv.index]; pv.columns = [SHORT[c] for c in pv.columns]
print(pv.to_string())

rows = []
for grp, g in C.groupby('kelompok'):
    sig = g[g.bermakna_logit | g.bermakna_prob]
    rows.append({'kelompok': grp, 'korelasi_r': np.corrcoef(g.r_logit, g.r_prob)[0, 1], 'selisih_r_terbesar': (g.r_logit - g.r_prob).abs().max(),
                 'ukuran_bermakna': len(sig), 'arah_sama': int((np.sign(sig.r_logit) == np.sign(sig.r_prob)).sum()),
                 'bermakna_di_keduanya': int((g.bermakna_logit & g.bermakna_prob).sum())})
K4 = pd.DataFrame(rows).set_index('kelompok').reindex(MAIN)
print('\nK4. KESEPAKATAN HASIL peta logit dan peta lama, per kelompok')
print(K4.round(3).to_string())

# K5: karya manusia di uji lintas generator, dipisah per model (di langkah 5 keduanya digabung)
hm = df[df.lintas & (df.source == 'human') & df.benar].copy(); hm['kelompok2'] = 'manusia benar | ' + hm.model
lol = df[df.lintas & (df.y == 1) & (df.pred == 0)].copy(); lol['kelompok2'] = lol.source.map(SRC) + ' lolos | ' + lol.model
S5 = effects(pd.concat([hm, lol]), 'kelompok2')
S5['sel'] = [cell(a, b) for a, b in zip(S5.r, S5.bermakna)]
pv = S5.pivot(index='ukuran', columns='kelompok', values='sel').reindex(MEASURES); pv.index = [LABEL[i] for i in pv.index]
print('\nK5. UJI LINTAS GENERATOR dipisah per model (peta logit, 20%). model-LD = dilatih pada LD; model-SD = dilatih pada SD')
print(pv.to_string())
S5.to_csv(RESULTS / 'rq3_lintas_per_model.csv', index=False)

# D: baseline ukuran desain seluruh gambar
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
w = pd.read_csv(RESULTS / 'rq2_elemen_per_gambar.csv', usecols=['bagian', 'sample_id', 'metode', 'ambang'] + [f'whole_{m}' for m in MEASURES])
w = w[w.bagian.str.startswith('riwayat_disamakan') & (w.metode == 'occlusion') & np.isclose(w.ambang, 0.2)].drop_duplicates('sample_id')
w = w.join(manifest.set_index('sample_id')[['source', 'fold', 'y']], on='sample_id').reset_index(drop=True)
SETS = {'semua (11)': MEASURES, 'warna (3)': ['hue_entropy', 'saturation', 'colourfulness'],
        'nada (3)': ['brightness', 'contrast', 'gradient'],
        'garis dan tekstur (5)': ['edge_density', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']}
TASKS = {'AI vs manusia': ['human', 'latent_diffusion', 'stable_diffusion'], 'manusia vs LD': ['human', 'latent_diffusion'], 'manusia vs SD': ['human', 'stable_diffusion']}
def fit_pred(tr, te, feats):
    mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    cols = [f'whole_{m}' for m in feats]
    mdl.fit(tr[cols].fillna(tr[cols].median()), tr.y); return mdl.predict_proba(te[cols].fillna(tr[cols].median()))[:, 1]
rows = []
for sname, feats in SETS.items():
    row = {'ukuran': sname}
    for tname, srcs in TASKS.items():
        d = w[w.source.isin(srcs)]; aucs = []
        for k in range(5):
            aucs.append(roc_auc_score(d[d.fold == k].y, fit_pred(d[d.fold != k], d[d.fold == k], feats)))
        row[tname] = f'{np.mean(aucs):.3f} ({np.std(aucs):.3f})'
    for a, b in (('latent_diffusion', 'stable_diffusion'), ('stable_diffusion', 'latent_diffusion')):
        tr = w[((w.source == 'human') & (w.fold != 4)) | (w.source == a)]; te = w[((w.source == 'human') & (w.fold == 4)) | (w.source == b)]
        p = fit_pred(tr, te, feats)
        row[f'latih {SRC[a]} uji {SRC[b]}: AUC'] = f'{roc_auc_score(te.y, p):.3f}'
        row[f'latih {SRC[a]} uji {SRC[b]}: AI dikenali'] = f'{(p[te.y.values == 1] >= 0.5).mean():.3f}'
    rows.append(row)
D = pd.DataFrame(rows).set_index('ukuran'); D.to_csv(RESULTS / 'rq2_baseline_ukuran_desain.csv')
print('\nD. BASELINE UKURAN DESAIN: regresi logistik pada ukuran seluruh gambar (AUC, 5-fold; dalam kurung simpangan baku)')
print(D.iloc[:, :3].to_string())
print('\nD2. BASELINE UKURAN DESAIN, uji lintas generator')
print(D.iloc[:, 3:].to_string())
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tertulis "Bagian selesai: 7 dari 7" dan tabel K1 sampai K5, W, D, dan D2 muncul, semua beres. Tekan **Cmd+S**, lalu kabari Claude.